# 05 - MODEL OPTIMIZATION

## 1 - IMPORT + COMMON

In [1]:
import torch
from torch.utils.data import TensorDataset
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
from sklearn.metrics import roc_auc_score, accuracy_score
import numpy as np

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Usando dispositivo:", device)
%env CUDA_LAUNCH_BLOCKING=1


Usando dispositivo: cuda
env: CUDA_LAUNCH_BLOCKING=1


In [3]:
from google.colab import drive
drive.mount('/content/drive')
url_pref="drive/MyDrive/Colab Notebooks/"

Mounted at /content/drive


In [4]:
class ECGNet_v2(nn.Module):
    def __init__(self, input_channels=12):
        super(ECGNet_v2, self).__init__()

        self.cnn = nn.Sequential(
            # Bloque 1
            nn.Conv1d(input_channels, 32, kernel_size=7, padding=3),
            nn.ReLU(),
            nn.BatchNorm1d(32),
            nn.MaxPool1d(2),
            nn.Dropout(0.2),

            # Bloque 2
            nn.Conv1d(32, 64, kernel_size=5, padding=2),
            nn.ReLU(),
            nn.BatchNorm1d(64),
            nn.MaxPool1d(2),
            nn.Dropout(0.3),

            # Bloque 3
            nn.Conv1d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.BatchNorm1d(128),
            nn.MaxPool1d(2),
            nn.Dropout(0.3),

            # Bloque 4 extra (más capacidad)
            nn.Conv1d(128, 256, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.BatchNorm1d(256),
            nn.AdaptiveAvgPool1d(1),  # colapsa la dimensión temporal
            nn.Dropout(0.4)
        )

        # Clasificador final
        self.fc = nn.Sequential(
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(64, 1)  # salida sin sigmoid
        )

    def forward(self, x):
        x = self.cnn(x)
        x = x.view(x.size(0), -1)  # aplanar [batch, channels]
        x = self.fc(x)
        return x  # salida sin sigmoid → usar BCEWithLogitsLoss



In [5]:
# Crear DataLoader usando señales y labels binarios (0/1) para entrenamiento desde archivos chunked zip
import torch
from torch.utils.data import TensorDataset, DataLoader
import zipfile
import os
import tempfile

def load_signals_from_zip(zip_path):
    signals_list, labels_list = [], []
    with zipfile.ZipFile(zip_path, 'r') as zipf:
        pt_files = [f for f in zipf.namelist() if f.endswith('.pt')]
        with tempfile.TemporaryDirectory() as tmpdir:
            for pt_file in pt_files:
                zipf.extract(pt_file, tmpdir)
                pt_path = os.path.join(tmpdir, pt_file)
                data = torch.load(pt_path)
                signals_list.append(data['signals'])
                labels_list.append(data['labels'])
    signals = torch.cat(signals_list, dim=0)
    labels = torch.cat(labels_list, dim=0)
    return signals, labels

# Cargar los datos preparados desde los zip chunked
train_signals, train_labels = load_signals_from_zip(url_pref+'TORCH/train_data_chunks.zip')
val_signals, val_labels = load_signals_from_zip(url_pref+'TORCH/val_data_chunks.zip')
test_signals, test_labels = load_signals_from_zip(url_pref+'TORCH/test_data_chunks.zip')

# Asegurarse que los labels sean float y binarios (0/1)
train_labels = train_labels.float().clamp(0, 1)
val_labels = val_labels.float().clamp(0, 1)
test_labels = test_labels.float().clamp(0, 1)

# Crear TensorDataset y DataLoader
train_dataset = TensorDataset(train_signals, train_labels)
val_dataset = TensorDataset(val_signals, val_labels)
test_dataset = TensorDataset(test_signals, test_labels)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

print('Ejemplo de batch de señales:', next(iter(train_loader))[0].shape)
print('Ejemplo de batch de labels:', next(iter(train_loader))[1][:10])

Ejemplo de batch de señales: torch.Size([16, 12, 2500])
Ejemplo de batch de labels: tensor([0., 0., 0., 0., 0., 0., 0., 0., 1., 0.])


## 2 - OPTIMIZER

In [6]:
param_grid = {
    "lr": [1e-3, 1e-4, 1e-5],
    "dropout_fc": [0.3, 0.5, 0.7],
    "batch_size": [32, 64, 128]
}


In [7]:
best_auc = 0
best_params = {}
for lr in param_grid["lr"]:
    for dropout_fc in param_grid["dropout_fc"]:
        # Crear modelo con nuevo dropout
        model = ECGNet_v2(input_channels=12).to(device)
        # Modificar dropout en fc
        for layer in model.fc:
            if isinstance(layer, nn.Dropout):
                layer.p = dropout_fc

        optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
        criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([19.0], device=device))

        # Entrenar solo unos pocos epochs para prueba rápida
        for epoch in range(3):
            model.train()
            for signals, labels in train_loader:
                signals, labels = signals.to(device).float(), labels.to(device).float().unsqueeze(1)
                optimizer.zero_grad()
                outputs = model(signals)
                loss = criterion(outputs, labels)
                loss.backward()
                optimizer.step()

        # Evaluación en validation
        model.eval()
        all_labels, all_preds = [], []
        with torch.no_grad():
            for signals, labels in val_loader:
                signals, labels = signals.to(device).float(), labels.to(device).float().unsqueeze(1)
                outputs = model(signals)
                probs = torch.sigmoid(outputs)
                all_labels.extend(labels.cpu().numpy().flatten())
                all_preds.extend(probs.cpu().numpy().flatten())

        val_auc = roc_auc_score(all_labels, all_preds)
        print(f"lr={lr}, dropout={dropout_fc} | Val AUC={val_auc:.4f}")

        if val_auc > best_auc:
            best_auc = val_auc
            best_params = {"lr": lr, "dropout_fc": dropout_fc}


lr=0.001, dropout=0.3 | Val AUC=0.9933
lr=0.001, dropout=0.5 | Val AUC=0.9894
lr=0.001, dropout=0.7 | Val AUC=0.9829
lr=0.0001, dropout=0.3 | Val AUC=0.9865
lr=0.0001, dropout=0.5 | Val AUC=0.9684
lr=0.0001, dropout=0.7 | Val AUC=0.9415
lr=1e-05, dropout=0.3 | Val AUC=0.7974
lr=1e-05, dropout=0.5 | Val AUC=0.7980
lr=1e-05, dropout=0.7 | Val AUC=0.7748


In [11]:
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    precision_score, recall_score, f1_score
)

def evaluate_metrics(y_true, y_pred_prob, threshold=0.5):
    y_pred = (y_pred_prob >= threshold).astype(int)
    return {
        "AUROC": roc_auc_score(y_true, y_pred_prob),
        "AUPRC": average_precision_score(y_true, y_pred_prob),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1": f1_score(y_true, y_pred, zero_division=0),
    }

best_params = {}
best_metrics = {"AUPRC": 0.0, "F1": 0.0}  # Criterio principal y secundario

for lr in param_grid["lr"]:
    for dropout_fc in param_grid["dropout_fc"]:

        # Crear modelo con nuevo dropout
        model = ECGNet_v2(input_channels=12).to(device)

        # Modificar dropout en la fully-connected
        for layer in model.fc:
            if isinstance(layer, nn.Dropout):
                layer.p = dropout_fc

        optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
        criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([19.0], device=device))

        # Entrenar solo 3 epochs como búsqueda rápida
        for epoch in range(3):
            model.train()
            for signals, labels in train_loader:
                signals = signals.to(device).float()
                labels = labels.to(device).float().unsqueeze(1)

                optimizer.zero_grad()
                outputs = model(signals)
                loss = criterion(outputs, labels)
                loss.backward()
                optimizer.step()

        # ----------------------------
        # Evaluación en validation set
        # ----------------------------
        model.eval()
        all_labels, all_preds = [], []

        with torch.no_grad():
            for signals, labels in val_loader:
                signals = signals.to(device).float()
                labels = labels.to(device).float().unsqueeze(1)

                outputs = model(signals)
                probs = torch.sigmoid(outputs)

                all_labels.extend(labels.cpu().numpy().flatten())
                all_preds.extend(probs.cpu().numpy().flatten())

        # Calcular métricas
        metrics = evaluate_metrics(
            np.array(all_labels),
            np.array(all_preds)
        )

        print(f"lr={lr}, dropout={dropout_fc} | "
              f"AUROC={metrics['AUROC']:.4f} | "
              f"AUPRC={metrics['AUPRC']:.4f} | "
              f"F1={metrics['F1']:.4f}")

        # ----------------------------
        # Selección del mejor modelo
        # ----------------------------
        if (
            metrics["AUPRC"] > best_metrics["AUPRC"] or
            (metrics["AUPRC"] == best_metrics["AUPRC"] and metrics["F1"] > best_metrics["F1"])
        ):
            best_metrics = metrics
            best_params = {"lr": lr, "dropout_fc": dropout_fc}


lr=0.001, dropout=0.3 | AUROC=0.9943 | AUPRC=0.9141 | F1=0.7370
lr=0.001, dropout=0.5 | AUROC=0.9921 | AUPRC=0.9017 | F1=0.7242
lr=0.001, dropout=0.7 | AUROC=0.9878 | AUPRC=0.8731 | F1=0.7000
lr=0.0001, dropout=0.3 | AUROC=0.9782 | AUPRC=0.8728 | F1=0.2755
lr=0.0001, dropout=0.5 | AUROC=0.9687 | AUPRC=0.7939 | F1=0.4268
lr=0.0001, dropout=0.7 | AUROC=0.9805 | AUPRC=0.8429 | F1=0.3030
lr=1e-05, dropout=0.3 | AUROC=0.8260 | AUPRC=0.2396 | F1=0.2338
lr=1e-05, dropout=0.5 | AUROC=0.8136 | AUPRC=0.2147 | F1=0.2793
lr=1e-05, dropout=0.7 | AUROC=0.7873 | AUPRC=0.1802 | F1=0.2243
